## Data Pipeline

In [6]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("hyperparam_tuning").config("spark.sql.shuffle.partitions", "4000").config("spark.executor.memoryOverhead", "4g").getOrCreate()

In [7]:
from datetime import datetime, timedelta
from pyspark.sql import functions as F

date = "2026-03-22"
train_days = 90
test_days = 30

daily_watch_history_path = "gs://wynk-ml-workspace/projects/rails_reranking/daily_user_watch_history_new"
date_format = "%Y-%m-%d"
base_date = datetime.strptime(date, date_format)

# 1. Test Paths: The last 30 days (0 to 29 days ago)
test_paths = [
    f"{daily_watch_history_path}/day={(base_date - timedelta(days=i)).strftime(date_format)}"
    for i in range(test_days)
]

# 2. Train Paths: The 90 days before the test period (30 to 119 days ago)
train_paths = [
    f"{daily_watch_history_path}/day={(base_date - timedelta(days=i)).strftime(date_format)}"
    for i in range(test_days, test_days + train_days)
]

test_df = None
train_df = None

for path in test_paths:
    try:
        temp_df = spark.read.parquet(path)

        if test_df is None:
            test_df = temp_df
        else:
            test_df = test_df.unionByName(temp_df)

    except Exception:
        print(f"Skipping missing TEST path: {path}")

for path in train_paths:
    try:
        temp_df = spark.read.parquet(path)

        if train_df is None:
            train_df = temp_df
        else:
            train_df = train_df.unionByName(temp_df)

    except Exception:
        print(f"Skipping missing TRAIN path: {path}")

Skipping missing TEST path: gs://wynk-ml-workspace/projects/rails_reranking/daily_user_watch_history_new/day=2026-03-22
Skipping missing TRAIN path: gs://wynk-ml-workspace/projects/rails_reranking/daily_user_watch_history_new/day=2026-02-06
Skipping missing TRAIN path: gs://wynk-ml-workspace/projects/rails_reranking/daily_user_watch_history_new/day=2026-02-05


In [8]:
enriched_db_path = "gs://wynk-ml-workspace/projects/xstream_nlu/catalog-db/" 

enriched_tv_df = spark.read.parquet(f'{enriched_db_path}{date}/enriched_tv.parquet')
enriched_movies_df = spark.read.parquet(f'{enriched_db_path}{date}/enriched_movie.parquet')

filtered_enriched_tv = (
    enriched_tv_df
    # 1. Remove empty arrays and un-published content
    .filter((F.col('XstreamContentIds') != F.array()) & (F.col("published") == True))
    # 2. Explode and select in one go
    .select(
        F.explode("XstreamContentIds").alias("item_id"), # This is your primary key for ALS joins
        "title", 
        "ID",
        F.col('OriginalLanguage').alias('original_language'), 
        "Genres"
    )
)
filtered_enriched_movies = (
    enriched_movies_df
    # 1. Remove empty arrays and un-published content
    .filter((F.col('XstreamContentIds') != F.array()) & (F.col("published") == True))
    # 2. Explode and select in one go
    .select(
        F.explode("XstreamContentIds").alias("item_id"), # This is your primary key for ALS joins
        "title", 
        "ID", 
        F.col('OriginalLanguage').alias('original_language'),
        "Genres"
    )
)

# 1. Ignore files that are physically broken/incomplete
spark.conf.set("spark.sql.files.ignoreCorruptFiles", "true")

# 2. Disable the Vectorized Reader (Standardizes the reading process)
spark.conf.set("spark.sql.parquet.enableVectorizedReader", "false")

# 3. Handle schema mismatches if they exist
spark.conf.set("spark.sql.parquet.mergeSchema", "true")

# 4. Perform the union with explicit casting to ensure types match
tv_final = filtered_enriched_tv.select(
    F.col("item_id").cast("string"),
    F.col("title").cast("string"),
    F.col("original_language").cast("string"),
    F.col("Genres").cast("string") # Ensure both are cast to the same type
)

movies_final = filtered_enriched_movies.select(
    F.col("item_id").cast("string"),
    F.col("title").cast("string"),
    F.col("original_language").cast("string"),
    F.col("Genres").cast("string")
)

enriched_content_df = tv_final.unionByName(movies_final).distinct()
# enriched_content_df.cache().count() # Use count() to force Spark to read and verify all files now
enriched_content_df = filtered_enriched_tv.unionByName(filtered_enriched_movies)

26/03/23 04:29:17 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [9]:
# 2. Get unique users (this is a heavy operation, so we do it once)
# Using .distinct() on 500GB will trigger a shuffle.
train_users = train_df.select("userId").distinct()
test_users = test_df.select("userId").distinct()

# 3. Join them to find the 'overlap'
common_users = train_users.join(test_users, on="userId", how="inner")

# 4. Filter the main dataframes
# We use an inner join here because it's generally more 
# efficient than 'isin' in Spark for large datasets.
df_train_filtered = train_df.join(common_users, on="userId", how="inner")
df_test_filtered = test_df.join(common_users, on="userId", how="inner")

In [10]:
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from pyspark.sql.functions import row_number

# 1. Aggregate Playtime and Count Distinct items per user in one flow
def compute_user_stats(watch_history_df):
    user_stats = (
        watch_history_df  
        .groupBy("userId", "item_id")
        .agg(
            F.sum("total_play_time_sec").alias("total_playtime_combined")
        )
        # We add a column to count how many distinct items EACH user has watched
        .withColumn("distinct_content_count", F.count("item_id").over(Window.partitionBy("userId")))
        .filter(
    F.col("total_playtime_combined").isNotNull() & 
    ~F.isnan("total_playtime_combined")
)
    )
    return user_stats

train_combined_stats = compute_user_stats(df_train_filtered)

# 2. Filter for users with at least 3 distinct items
als_input_base = train_combined_stats.filter("distinct_content_count >= 2")

# als_input_base.cache()
# 3. Join them to find the 'overlap'
re_common_users = test_users.join(als_input_base, on="userId", how="inner").cache()

# 4. Filter the main dataframes
# We use an inner join here because it's generally more 
# efficient than 'isin' in Spark for large datasets.
df_test_filtered = df_test_filtered.join(re_common_users, on="userId", how="inner")

import pyspark.sql.functions as F
from pyspark.ml.feature import StringIndexer
from pyspark.ml import Pipeline
from pyspark.ml.recommendation import ALS

# 1. Get distinct IDs
distinct_users = als_input_base.select("userId").distinct()
distinct_items = als_input_base.select("item_id").distinct()

# 2. Use RDD zipWithIndex for highly scalable, distributed ID generation
user_lookup = (
    distinct_users.rdd.zipWithIndex()
    .toDF(["user_struct", "userIndex"])
    .select("user_struct.*", "userIndex")
)

item_lookup = (
    distinct_items.rdd.zipWithIndex()
    .toDF(["item_struct", "itemIndex"])
    .select("item_struct.*", "itemIndex")
)

# 3. Join back to the main data
indexed_data = (
    als_input_base
    .join(user_lookup, on="userId", how="inner")
    .join(item_lookup, on="item_id", how="inner")
)

als_data = (
    indexed_data
    .withColumn("playtime_logged", F.log1p("total_playtime_combined"))
    # The .cast("int") strips the heavy StringIndexer metadata out of the schema!
    .select(
        F.col("userIndex").cast("int").alias("userIndex"), 
        F.col("itemIndex").cast("int").alias("itemIndex"), 
        "playtime_logged"
    )
    .repartition(1000) 
)

df_train_indexed = indexed_data

df_test_indexed = (
    df_test_filtered
    .join(user_lookup, on="userId", how="inner")
    .join(item_lookup, on="item_id", how="inner")
)

ground_truth = (
    df_test_indexed
    .groupBy("userIndex")
    .agg(F.collect_set("itemIndex").alias("actual_items"))
)

In [11]:
# 4. Save lookups to disk immediately to break the lineage!
user_lookup.write.mode("overwrite").parquet("gs://wynk-ml-workspace/_temp/harshith/als/user_lookup")
item_lookup.write.mode("overwrite").parquet("gs://wynk-ml-workspace/_temp/harshith/als/item_lookup")

In [12]:
user_lookup = spark.read.parquet("gs://wynk-ml-workspace/_temp/harshith/als/user_lookup")
item_lookup = spark.read.parquet("gs://wynk-ml-workspace/_temp/harshith/als/item_lookup")

In [14]:
# 5. Train the Model
als_200 = ALS(
    userCol="userIndex",
    itemCol="itemIndex",
    ratingCol="playtime_logged",
    implicitPrefs=True,
    rank=200,
    maxIter=10,
    regParam=0.1,
    coldStartStrategy="drop"
)

model_200 = als_200.fit(als_data)
print("Model trained with rank=200")
als_model_path = "gs://wynk-ml-workspace/_temp/harshith/als/als_model_200"
model_200.write().overwrite().save(als_model_path)

# 1. Generate top 15 recommendations for all users
# Note: This returns a DataFrame with: [userIndex, recommendations: array<struct<itemIndex, rating>>]
user_recs = model_200.recommendForAllUsers(15)

# 2. Flatten the recommendations to make it easier to compare
from pyspark.sql import functions as F

user_recs_flat = user_recs.select(
    F.col("userIndex"),
    F.col("recommendations.itemIndex").alias("predicted_items")
)
from pyspark.mllib.evaluation import RankingMetrics

# 1. Join predictions with ground truth
eval_data = user_recs_flat.join(ground_truth, on="userIndex", how="inner")

# 2. Convert to RDD of (predicted_array, actual_array)
# RankingMetrics expects a list of items, not a list of Rows
prediction_and_labels = eval_data.select("predicted_items", "actual_items").rdd.map(tuple)

# 3. Instantiate RankingMetrics
metrics = RankingMetrics(prediction_and_labels)

# 4. Extract results
print(f"Precision at 15: {metrics.precisionAt(15):.4f}")
print(f"Recall at 15:    {metrics.recallAt(15):.4f}")
print(f"Mean Average Precision (MAP): {metrics.meanAveragePrecision:.4f}")


26/03/23 04:30:28 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/23 04:30:30 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/23 04:30:32 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/23 04:30:41 WARN SparkConf: The configuration key 'spark.yarn.executor.failuresValidityInterval' has been deprecated as of Spark 3.5 and may be removed in the future. Please use the new key 'spark.executor.failuresValidityInterval' instead.
26/03/23 04:

Precision at 15: 0.0241


26/03/23 05:04:52 WARN DAGScheduler: Broadcasting large task binary with size 5.5 MiB


Recall at 15:    0.1004


Mean Average Precision (MAP): 0.0496


## LightFM

In [ ]:
import pandas as pd
import numpy as np
import ast
import pyspark.sql.functions as F
from pyspark.sql.window import Window
from lightfm.data import Dataset
from lightfm import LightFM
from lightfm.evaluation import precision_at_k, recall_at_k

# =====================================================================
# 1. SPARK: AGGREGATE TRAIN DATA
# =====================================================================
print("Aggregating Spark Train Data...")
def compute_user_stats(watch_history_df):
    user_stats = (
        watch_history_df  
        .groupBy("userId", "item_id")
        .agg(F.sum("total_play_time_sec").alias("total_playtime_combined"))
        .withColumn("distinct_content_count", F.count("item_id").over(Window.partitionBy("userId")))
        .filter(F.col("total_playtime_combined").isNotNull() & ~F.isnan("total_playtime_combined"))
    )
    return user_stats

train_combined_stats = compute_user_stats(df_train_filtered)

# Filter for users with at least 2 distinct items
als_input_base = train_combined_stats.filter("distinct_content_count >= 2").cache()

# =====================================================================
# 2. SPARK: FIND OVERLAPPING USERS & FILTER TEST DATA
# =====================================================================
print("Filtering Test Data for Overlapping Users...")
test_users = df_test_filtered.select("userId").distinct()
re_common_users = test_users.join(als_input_base.select("userId").distinct(), on="userId", how="inner").cache()

df_test_filtered = df_test_filtered.join(re_common_users, on="userId", how="inner")

# =====================================================================
# 3. SPARK: CREATE CUSTOM LOOKUPS & APPLY TO TRAIN/TEST
# =====================================================================
print("Creating RDD Lookups and Indexing Data...")
distinct_users = als_input_base.select("userId").distinct()
distinct_items = als_input_base.select("item_id").distinct()

user_lookup = (
    distinct_users.rdd.zipWithIndex()
    .toDF(["user_struct", "userIndex"])
    .select(F.col("user_struct.*"), F.col("userIndex").cast("int"))
)

item_lookup = (
    distinct_items.rdd.zipWithIndex()
    .toDF(["item_struct", "itemIndex"])
    .select(F.col("item_struct.*"), F.col("itemIndex").cast("int"))
)

# Apply lookups to Train
df_train_indexed = (
    als_input_base
    .join(user_lookup, on="userId", how="inner")
    .join(item_lookup, on="item_id", how="inner")
    .withColumn("playtime_logged", F.log1p("total_playtime_combined"))
)

# Apply lookups to Test
df_test_indexed = (
    df_test_filtered
    .join(user_lookup, on="userId", how="inner")
    .join(item_lookup, on="item_id", how="inner")
    # Assuming test data also has 'total_play_time_sec' to be logged
    .withColumn("playtime_logged", F.log1p("total_play_time_sec")) 
)

# =====================================================================
# 3.5 SPARK: SCRUB TEST DATA OF ALREADY-SEEN ITEMS
# =====================================================================
print("Removing train user-item pairs from the test set...")

# A left_anti join drops any row in df_test_indexed where the exact 
# same userIndex and itemIndex pair already exists in df_train_indexed.
df_test_indexed = df_test_indexed.join(
    df_train_indexed.select("userIndex", "itemIndex"),
    on=["userIndex", "itemIndex"],
    how="left_anti"
)

# =====================================================================
# 4. SPARK TO PANDAS: SAFE SAMPLING FOR LIGHTFM
# =====================================================================
print("Sampling Data for LightFM (to prevent Driver OOM)...")
# WARNING: LightFM runs in memory on the driver. We MUST sample the 500GB data.
# We sample 5% of users. Adjust this fraction based on your driver's RAM.
sampled_users = re_common_users.sample(withReplacement=False, fraction=0.05, seed=42)

sampled_train_spark = df_train_indexed.join(sampled_users, on="userId", how="inner")
sampled_test_spark = df_test_indexed.join(sampled_users, on="userId", how="inner")

print("Fetching interactions from Spark to Pandas...")
train_pdf = sampled_train_spark.select("userIndex", "itemIndex", "playtime_logged").toPandas()
test_pdf = sampled_test_spark.select("userIndex", "itemIndex", "playtime_logged").toPandas()

print("Fetching items metadata from Spark to Pandas...")
items_pdf = (
    enriched_content_df.join(item_lookup, "item_id")
    .select("itemIndex", "item_id", "title", "original_language", "Genres")
    .dropDuplicates(["itemIndex"])
    .toPandas()
)

# =====================================================================
# 5. PANDAS: CLEAN DATA & FEATURE EXTRACTION
# =====================================================================
print("Cleaning missing values and building features...")
items_pdf['Genres'] = items_pdf['Genres'].fillna('Unknown').astype(str)
items_pdf['original_language'] = items_pdf['original_language'].fillna('Unknown').astype(str)

# =====================================================================
# FILTER ORPHAN ITEMS OUT OF METADATA
# =====================================================================
print("Filtering orphan items from metadata...")

# Find all unique items present in our sampled train and test sets
valid_sampled_items = np.unique(np.concatenate((train_pdf['itemIndex'], test_pdf['itemIndex'])))

# Filter the items_pdf to only include these valid items
items_pdf = items_pdf[items_pdf['itemIndex'].isin(valid_sampled_items)]

print(f" -> Reduced items metadata to {len(items_pdf)} valid items.")

def extract_flat_features(lang, genre_data):
    features = [str(lang)]
    if isinstance(genre_data, str):
        if genre_data.startswith('['):
            try:
                genre_data = ast.literal_eval(genre_data)
            except:
                genre_data = genre_data.replace('[', '').replace(']', '').replace("'", "").replace('"', "")
                genre_data = [g.strip() for g in genre_data.split(',')]
        else:
            genre_data = [g.strip() for g in genre_data.split(',')]
    if isinstance(genre_data, list):
        for item in genre_data:
            if isinstance(item, list):
                features.extend([str(i).strip() for i in item])
            else:
                features.append(str(item).strip())
    return features

items_pdf['clean_features'] = items_pdf.apply(
    lambda row: extract_flat_features(row['original_language'], row['Genres']), axis=1
)

all_item_features = list(set([feat for sublist in items_pdf['clean_features'] for feat in sublist]))

# =====================================================================
# 6. LIGHTFM: INITIALIZE DATASET & BUILD MATRICES
# =====================================================================
print("Fitting LightFM Dataset mapping...")
# CRITICAL: Fit the dataset on the combined unique IDs from train and test
all_users = np.unique(np.concatenate((train_pdf['userIndex'], test_pdf['userIndex'])))
all_items = np.unique(np.concatenate((train_pdf['itemIndex'], test_pdf['itemIndex'])))

dataset = Dataset()
dataset.fit(
    users=all_users,
    items=all_items,
    item_features=all_item_features
)

print("Building Interactions matrices...")
(train_interactions, train_weights) = dataset.build_interactions(
    zip(train_pdf['userIndex'], train_pdf['itemIndex'], train_pdf['playtime_logged'])
)

(test_interactions, _) = dataset.build_interactions(
    zip(test_pdf['userIndex'], test_pdf['itemIndex'], test_pdf['playtime_logged'])
)

print("Building Item Features matrix...")
item_features = dataset.build_item_features(
    (item_idx, features) for item_idx, features in zip(items_pdf['itemIndex'], items_pdf['clean_features'])
)

print("cell executed successfully!")



In [ ]:
# =====================================================================
# 7. LIGHTFM: TRAIN & EVALUATE
# =====================================================================
print("Training LightFM model...")
lfm_model = LightFM(
    loss='warp',
    no_components=20,
    learning_rate=0.05,
    item_alpha=1e-6,
    user_alpha=1e-6
)

lfm_model.fit(
    train_interactions,
    item_features=item_features,
    sample_weight=train_weights,
    epochs=10,
    num_threads=4 
)
print("Training complete!")

In [ ]:
import scipy.sparse as sp

print("Evaluating Model (Masking out training items)...")

# =====================================================================
# THE FIX: SCRUB THE SPARSE MATRIX DIRECTLY
# =====================================================================
# Convert to CSR (Compressed Sparse Row) format for fast arithmetic
test_csr = test_interactions.tocsr()
train_csr = train_interactions.tocsr()

# Find the exact overlap (where test has values AND train has values)
overlap = test_csr.multiply(train_csr.astype(bool))

# Subtract the overlap from the test matrix to leave ONLY completely new interactions
clean_test_interactions = (test_csr - overlap).tocoo()
print(f"Removed {overlap.nnz} overlapping interactions from the test set.")
# =====================================================================

test_precision = precision_at_k(
    lfm_model, 
    clean_test_interactions,               # <--- USE THE CLEANED MATRIX HERE
    train_interactions=train_interactions, # This still masks out train items during ranking
    item_features=item_features, 
    k=15,
    num_threads=4
).mean()

test_recall = recall_at_k(
    lfm_model, 
    clean_test_interactions,               # <--- USE THE CLEANED MATRIX HERE
    train_interactions=train_interactions, # This still masks out train items during ranking
    item_features=item_features, 
    k=15,
    num_threads=4
).mean()

print(f"Test Precision@15: {test_precision:.4f}")
print(f"Test Recall@15:    {test_recall:.4f}")